# 02 — Pixel-wise Random Forest and SVM

Before U-Nets, DRIVE papers treated **each FOV pixel as a sample**. The feature vector mixed intensity, local statistics, gradients and a vesselness filter. A Random Forest or SVM then classified vessel vs background.

Class imbalance is severe (~10% vessel pixels). We **undersample** the majority class when fitting, and keep `class_weight="balanced"`. The SVM here is **linear** (`LinearSVC`): an RBF SVM on every pixel is too slow for a teaching demo and is not what this notebook claims to be.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parents[0] / "src") if Path.cwd().name == "notebooks" else "src")

import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from drive_seg.data import iter_samples, load_sample, resolve_dataset_root
from drive_seg.metrics import segmentation_metrics
from drive_seg.ml import (
    FEATURE_NAMES,
    build_random_forest,
    build_svm,
    extract_features,
    pixels_from_sample,
    predict_mask,
)
from drive_seg.synthetic import generate_synthetic_dataset
from drive_seg.paths import SYNTHETIC_DIR

if not (SYNTHETIC_DIR / "training" / "images").is_dir():
    generate_synthetic_dataset()
root = resolve_dataset_root()
rng = np.random.default_rng(0)

## Feature volume

`extract_features` returns an `H×W×C` stack. Visualising a few channels on one test image shows why Frangi is included *as a feature* rather than as the whole method: the forest can ignore it on the optic disc, where vesselness often fires on the rim.

In [ ]:
sample = iter_samples(root, split="test")[0]
image, vessel, fov = load_sample(sample)
feats = extract_features(image)
print("feature volume", feats.shape, "names", FEATURE_NAMES)

show = [0, 6, 7]  # intensity, frangi, hessian ridge
fig = make_subplots(rows=1, cols=3, subplot_titles=[FEATURE_NAMES[i] for i in show])
for col, i in enumerate(show, start=1):
    fig.add_trace(go.Heatmap(z=feats[..., i], colorscale="Viridis", showscale=False), 1, col)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
fig.update_layout(template="plotly_white", height=340, title="Hand-crafted features")
fig.show()

## Fit RF and linear SVM

In [ ]:
xs, ys = [], []
for s in iter_samples(root, split="training"):
    img, gt, f = load_sample(s)
    x, y = pixels_from_sample(img, gt, f, rng, max_per_class=4000)
    xs.append(x); ys.append(y)
X, y = np.concatenate(xs), np.concatenate(ys)
print("train pixels", X.shape, "positive rate", float(y.mean()))

rf = build_random_forest()
svm = build_svm()
rf.fit(X, y)
svm.fit(X, y)
print("RF feature importances:")
for name, imp in sorted(zip(FEATURE_NAMES, rf.feature_importances_), key=lambda t: -t[1]):
    print(f"  {name:16s} {imp:.3f}")

## Test comparison

Expect RF ≥ linear SVM ≥ chance, and usually RF ≥ Frangi, because Frangi is one of the input channels. If RF importance on `frangi` is dominant, the forest is mostly learning a fancy threshold — which is still a fair 2012-era DRIVE result.

In [ ]:
rows = []
pred_rf = pred_svm = None
for s in iter_samples(root, split="test"):
    img, gt, f = load_sample(s)
    pr = predict_mask(rf, img, f)
    ps = predict_mask(svm, img, f)
    for method, p in ("rf", pr), ("svm", ps):
        r = segmentation_metrics(p, gt, f)
        r.update(method=method, sample_id=s.sample_id)
        rows.append(r)
    if pred_rf is None:
        pred_rf, pred_svm, image, vessel, fov = pr, ps, img, gt, f

df = pd.DataFrame(rows)
print(df.groupby("method")[["dice", "iou", "sensitivity", "specificity", "precision"]].mean().round(3))
mean = df.groupby("method").mean(numeric_only=True).reset_index()
long_m = mean.melt(
    id_vars=["method"],
    value_vars=["dice", "iou", "sensitivity", "precision"],
    var_name="metric",
    value_name="value",
)
fig = px.bar(
    long_m,
    x="metric",
    y="value",
    color="method",
    barmode="group",
    range_y=[0, 1],
    title="Pixel classifiers on the demo test split",
)
fig.update_layout(template="plotly_white")
fig.show()


In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("ground truth", "Random Forest", "linear SVM"))
fig.add_trace(go.Heatmap(z=vessel.astype(float), colorscale="Gray", showscale=False), 1, 1)
fig.add_trace(go.Heatmap(z=pred_rf.astype(float), colorscale="Gray", showscale=False), 1, 2)
fig.add_trace(go.Heatmap(z=pred_svm.astype(float), colorscale="Gray", showscale=False), 1, 3)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
fig.update_layout(template="plotly_white", height=340)
fig.show()